# 2.nivelSegmento · 3. Benchmark de modelos (nivel de ventana → sesión)

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_fscore_support

from sklearn.linear_model import Lasso, ElasticNet
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from pathlib import Path

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print('xgboost no disponible -> se omite (pip install xgboost)')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PROJECT_DIR = Path.home().as_posix() + '/Desktop/Master/UNIR_IA_TFE'
path_df = PROJECT_DIR + '/output/df_segment_features_vf.csv'
PHQ_THRESHOLD = 10
RANDOM_STATE  = 42
N_CV_FOLDS    = 5
K_FEATURES    = 20
NON_FEATURE = ['participant_id', 'phq8_binary', 'phq8_score', 'split', 'window_idx']
TARGET, GROUP = 'phq8_score', 'participant_id'
ACOUSTIC_PREFIXES = ('f0_', 'energy_', 'mfcc', 'centroid_', 'bandwidth_',
                     'rolloff_', 'zcr_', 'hnr_', 'jitter_', 'shimmer_')

## 1. Datos, partición por sesión y CV por grupo

In [3]:
df = pd.read_csv(path_df)
COLS_FEATURES = [c for c in df.columns if c not in NON_FEATURE]
ACOUSTIC_COLS = [c for c in COLS_FEATURES if c.startswith(ACOUSTIC_PREFIXES)]

sess = df.drop_duplicates(GROUP).set_index(GROUP)
sess_bin = (sess[TARGET] >= PHQ_THRESHOLD).astype(int)

df_td = df[df['split'].isin(['train', 'dev'])].reset_index(drop=True)
df_te = df[df['split'] == 'test'].reset_index(drop=True)
X_td, y_td, g_td = df_td[COLS_FEATURES], df_td[TARGET], df_td[GROUP]
X_te, y_te, g_te = df_te[COLS_FEATURES], df_te[TARGET], df_te[GROUP]

y_td_bin = (y_td >= PHQ_THRESHOLD).astype(int)
sgkf = StratifiedGroupKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(sgkf.split(X_td, y_td_bin, groups=g_td))
for k, (tr, va) in enumerate(cv_splits):
    assert not (set(g_td.iloc[tr]) & set(g_td.iloc[va])), f"FUGA fold {k}"
print(f"Ventanas train+dev {len(X_td)} | test {len(X_te)} | sin fuga de sesiones: OK")

Ventanas train+dev 13794 | test 5215 | sin fuga de sesiones: OK


## 2. Pipeline, agregación y función de evaluación

In [4]:
class GenderZScoreScaler(BaseEstimator, TransformerMixin):
    def __init__(self, cols, gender_col='gender'):
        self.cols = cols; self.gender_col = gender_col
    def fit(self, X, y=None):
        self.stats_ = {}
        for g, sub in X.groupby(self.gender_col):
            self.stats_[g] = (sub[self.cols].mean(), sub[self.cols].std(ddof=0).replace(0, 1.0))
        self.global_ = (X[self.cols].mean(), X[self.cols].std(ddof=0).replace(0, 1.0))
        return self
    def transform(self, X):
        X = X.copy()
        for g, idx in X.groupby(self.gender_col).groups.items():
            mu, sd = self.stats_.get(g, self.global_)
            X.loc[idx, self.cols] = (X.loc[idx, self.cols] - mu) / sd
        return X

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X): return X.drop(columns=[c for c in self.cols if c in X.columns])

def make_pipeline(model):
    return Pipeline([
        ('gender', GenderZScoreScaler(ACOUSTIC_COLS)),
        ('dropgender', DropColumns(['gender'])),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('select', SelectKBest(f_regression, k=K_FEATURES)),
        ('model', model),
    ])

def agg(part_ids, pred):
    return pd.Series(np.asarray(pred), index=np.asarray(part_ids)).groupby(level=0).mean()

def evaluate(model):
    pipe = make_pipeline(model)
    # OOF a nivel de sesión
    oof = {}
    for tr, va in cv_splits:
        m = clone(pipe); m.fit(X_td.iloc[tr], y_td.iloc[tr])
        oof.update(agg(g_td.iloc[va], m.predict(X_td.iloc[va])).to_dict())
    oof_p = pd.Series(oof); oof_y = sess_bin.loc[oof_p.index]
    auc_cv = roc_auc_score(oof_y, oof_p)
    ths = np.linspace(oof_p.min(), oof_p.max(), 200)
    thr = ths[int(np.argmax([f1_score(oof_y, (oof_p >= t).astype(int), zero_division=0) for t in ths]))]
    # test a nivel de sesión
    pipe.fit(X_td, y_td)
    ps = agg(g_te, pipe.predict(X_te)); ys = sess_bin.loc[ps.index]
    auc = roc_auc_score(ys, ps)
    rng = np.random.RandomState(RANDOM_STATE); yv, pv = ys.values, ps.values; boot = []
    for _ in range(2000):
        s = rng.randint(0, len(yv), len(yv))
        if len(np.unique(yv[s])) == 2: boot.append(roc_auc_score(yv[s], pv[s]))
    lo, hi = np.percentile(boot, [2.5, 97.5])
    p, r, f, _ = precision_recall_fscore_support(ys, (ps >= thr).astype(int),
                                                 average='binary', zero_division=0)
    return {'AUC_cv': auc_cv, 'AUC_test': auc, 'IC95': f"[{lo:.2f}, {hi:.2f}]",
            'Prec': p, 'Rec': r, 'F1_test': f}

## 3. Ejecución del benchmark

In [5]:
models = {
    'Lasso':        Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=10000),
    'ElasticNet':   ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000),
    'LinearSVR':    LinearSVR(C=1.0, random_state=RANDOM_STATE, max_iter=10000),
    'RandomForest': RandomForestRegressor(n_estimators=330, max_depth=8, min_samples_leaf=13,
                                          min_samples_split=33, max_features='log2',
                                          random_state=RANDOM_STATE, n_jobs=-1),
    'GradBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}
if HAS_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.03,
                                     subsample=0.8, colsample_bytree=0.8,
                                     random_state=RANDOM_STATE, n_jobs=-1)

rows = {}
for name, mdl in models.items():
    print(f"  · {name} ...", flush=True)
    rows[name] = evaluate(mdl)

results = pd.DataFrame(rows).T[['AUC_cv', 'AUC_test', 'IC95', 'Prec', 'Rec', 'F1_test']].round(3)
print(f"\nReferencias AUC test: RF sesión 0.743 | RF segmento 0.728")
results

  · Lasso ...
  · ElasticNet ...
  · LinearSVR ...
  · RandomForest ...
  · GradBoosting ...
  · XGBoost ...

Referencias AUC test: RF sesión 0.743 | RF segmento 0.728


,AUC_cv,AUC_test,IC95,Prec,Rec,F1_test
Lasso,0.602254,0.716518,"[0.54, 0.88]",0.363636,0.857143,0.510638
ElasticNet,0.601055,0.71875,"[0.55, 0.88]",0.393939,0.928571,0.553191
LinearSVR,0.568689,0.640625,"[0.45, 0.83]",0.32,0.571429,0.410256
RandomForest,0.654519,0.727679,"[0.57, 0.87]",0.428571,0.642857,0.514286
GradBoosting,0.608247,0.625,"[0.45, 0.79]",0.325,0.928571,0.481481
XGBoost,0.604411,0.622768,"[0.43, 0.80]",0.315789,0.857143,0.461538
